In [ ]:
import numpy as np
from numpy import array as arr
from numpy import deg2rad as d2r
from scipy.optimize import approx_fprime, minimize

from space_traj_opt.models.models import STANDARD_GRAV, CtrlMode
from space_traj_opt.optimization.phases import (
    DynEnum,
    Phase,
    PhaseDefect,
    TerminalConditions,
)
from space_traj_opt.optimization.problem import Problem
from space_traj_opt.optimization.transcription import MultiShootingTranscription
from space_traj_opt.optimization.utils import (
    denormalize_decision_vec,
    normalize_decision_vec,
)
from space_traj_opt.postprocessing.plotting import plot
from space_traj_opt.postprocessing.utils import unpack_sol_list


## Electron Rocket Parameters

In [ ]:
n_engines_s1 = 9
n_engines_s2 = 1
isp_s1 = 311.0
engine_thrust_s1 = n_engines_s1*24910.04  # N Average between sl and vac
isp_s2 = 343.0
engine_thrust_s2 = n_engines_s2* 25_000.0  # N
s1_vch_params = (engine_thrust_s1, isp_s1)
s2_vch_params = (engine_thrust_s2, isp_s2)

fairing_mass = 50.0
farinig_timing = 184.0 - 162.0 # sec
payload = 250.0
s1_dry_mass = 1076.47308279  
s2_dry_mass = 257.90093739  

s1_wet_mass = 10047.082106064723
s2_wet_mass = 2602.454913676189
total_mass = 12949.537019740912

mdot_s1 = engine_thrust_s1 / STANDARD_GRAV / isp_s1
mdot_s2 = engine_thrust_s2 / STANDARD_GRAV / isp_s2

In [ ]:
farinig_timing

In [ ]:
NUM_X= 5

# %load_ext snakeviz

## Initial Guesses

In [ ]:
mu_earth = 3.986004418e14
earth_r = 6_378_000.0 # m
circ_orbit_alt = 200_000.0 
v_circ = np.sqrt(mu_earth / (earth_r + circ_orbit_alt))

# Guesses 
s2_sep_mass = s2_wet_mass + payload + fairing_mass
x0 = arr([
    [0,0,0,0,total_mass], 
    [7.5,390,1.5,80,12200],
    [30000,60000,2000,500, s2_sep_mass],
    [45000,80000,2800,1000, s2_wet_mass + payload - farinig_timing * mdot_s2]
])

x_f = arr([1000000, circ_orbit_alt, v_circ, 0.0, s2_dry_mass + payload])

## normalization vector 
x0_n_vec = arr([circ_orbit_alt, circ_orbit_alt, 5000, 1000, 5000])

In [ ]:
s2_sep_mass

## Define a multiphase trajectory problem

In [ ]:
problem_builder = MultiShootingTranscription(["phase0", "phase1", "phase2", "phase3"], NUM_X)

## Define state, control and time guesses for each phase 

In [ ]:
state_bounds = [(0, None), (0, None), (0, None), (0, None), (100, None)]

phase0 = Phase("phase0", DynEnum.DYNAMICS_2D, CtrlMode.ANGLE_STEER ,s1_vch_params)
phase0.set_state(DynEnum.DYNAMICS_2D, x0[0], bounds=x0[0], norm_vec=x0_n_vec)
phase0.set_controller(
    CtrlMode.ANGLE_STEER,
    u0=d2r(89.5),
    bounds=[(d2r(85), d2r(89.8))],
    norm_vec=[np.pi],
)
phase0.set_time(10, bounds=10)

phase1 = Phase("phase1", DynEnum.DYNAMICS_2D, CtrlMode.ZERO_ALPHA, s1_vch_params)
phase1.set_state(DynEnum.DYNAMICS_2D, x0[1], bounds=state_bounds, norm_vec=x0_n_vec)
phase1.set_controller(CtrlMode.ZERO_ALPHA, u0=[], norm_vec=[])
phase1.set_time(100, bounds=(60, 180))

s2_sep_mass = s2_wet_mass + payload + fairing_mass
sep_bound = [(0, None), (0, None), (0, None), (0, None), (s2_sep_mass, s2_sep_mass)]
phase2 = Phase("phase2", DynEnum.DYNAMICS_2D, CtrlMode.LTS, s2_vch_params)
phase2.set_state(DynEnum.DYNAMICS_2D, x0[2], bounds=sep_bound, norm_vec=x0_n_vec)
phase2.set_controller(
    CtrlMode.LTS,
    u0=arr([-0.001, 1]),
    bounds=[(-0.1, 0.1), (-3, 3)],
    norm_vec=[0.1, np.pi / 2],
)
phase2.set_time(farinig_timing, bounds=farinig_timing)

phase3 = Phase("phase3", DynEnum.DYNAMICS_2D, CtrlMode.LTS, s2_vch_params)
phase3.set_state(DynEnum.DYNAMICS_2D, x0[3], norm_vec=x0_n_vec)
phase3.set_controller(
    CtrlMode.LTS,
    u0=arr([-0.001, 1]),
    bounds=[(-0.1, 0.1), (-3, 3)],
    norm_vec=[0.1, np.pi / 2],
)
phase3.set_time(320)

for phase in (phase0, phase1, phase2, phase3):
    problem_builder.add_phase(phase.name, phase)

problem_builder.add_defect(
    "stage_1_separation",
    ("phase1", "phase2"),
    PhaseDefect("stage_1_separation", arr([0, 0, 0, 0, s1_dry_mass]), arr([100000, 100000, 8000, 5000, 1000])),
)
problem_builder.add_defect(
    "fairing_separation",
    ("phase2", "phase3"),
    PhaseDefect("fairing_separation", arr([0, 0, 0, 0, fairing_mass]), arr([100000, 100000, 8000, 5000, 1000])),
)
problem_builder.add_terminal(
    TerminalConditions(
        x_final=x_f,
        bounds=[(None, None), (circ_orbit_alt, circ_orbit_alt), (v_circ, v_circ), (0, 0), (None, None)],
        norm_vec=x0_n_vec,
    )
)


## Build The problem
Builds the decision vector and bounds 

In [ ]:
d0, d_bounds, normalization_vec, full_params = problem_builder.build()
d0_norm, d_bounds_norm = normalize_decision_vec(
    d0,
    d_bounds,
    normalization_vec,
)

problem = Problem(
    d0_norm, 
    d_bounds_norm, 
    normalization_vec,
    5,5,4)

## Defining dynamic constraint function

In [ ]:
constraints = [{'type': 'eq', 'fun': problem.dynamics_knot_constrant, 'args':(full_params,) },]

## Scipy Minimize
SLSQP has to be used here because it can handle bounds and equality constraints.

In [ ]:
# %%snakeviz

result = minimize(
    problem.objective, 
    problem.d0_guess_normalized, 
    jac= problem.jac_objective,
    method='SLSQP', 
    bounds=problem.d_bounds_norm, 
    constraints=constraints,
    options = {"maxiter": 500, "disp": True},
    args=(full_params,)
)

In [ ]:
constraint_jac = approx_fprime(
    result.x, 
    problem.dynamics_knot_constrant, 
    np.float64(1.4901161193847656e-08), full_params)

In [ ]:
# visualize_jac2(result.x, constraint_jac)

In [ ]:
x_opt = denormalize_decision_vec(result.x, normalization_vec)

sol_list = problem.full_traj_rollout(x_opt, full_params)


In [ ]:
from space_traj_opt.postprocessing.post_proccess import sol_to_csv

name = "traj_opt_sol"
sol_to_csv(sol_list, ["x", "y", "vx", "vy", "m"], name)


In [ ]:
times, state= unpack_sol_list(sol_list,4)

In [ ]:
# lts_2 = lts_control(times[2] - times[2][0], 0, problem.unpack_decision_var(x_opt, full_params[2])[0])
# lts_3 = lts_control(times[3] - times[3][0], 0, problem.unpack_decision_var(x_opt, full_params[3])[0])


In [ ]:
# plot(
#     [times[2], times[3]],
#     [lts_2, lts_3],
#     title="Time vs States", 
#     xlabel="Time", 
#     ylabel="Pitch",
#     trace_names=("phase2", "phase3")
#     )